In [205]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import yfinance as yf
TATA_clean = pd.read_csv("TATASTEEL.csv")
TATA_clean=pd.DataFrame(TATA_clean)
print(TATA_clean)


    years     Sales    CAPEX  Working capital      PBT  Interest     EBIT  \
0  2021.0  156477.0  -9437.0          44327.0  13844.0    7607.0  21451.0   
1  2022.0  243959.0 -10905.0          44381.0  50227.0    5462.0  55689.0   
2  2023.0  243353.0 -18179.0          21683.0  18235.0    6299.0  24534.0   
3  2024.0  229171.0 -14253.0          20301.0  -1147.0    7508.0   6361.0   
4  2025.0  218543.0 -13611.0          23138.0   8413.0    7341.0  15754.0   
5     NaN       NaN      NaN              NaN      NaN       NaN      NaN   

   Unnamed: 7  Unnamed: 8  Unnamed: 9  Unnamed: 10  Depreciation  
0         NaN         NaN         NaN          NaN       9233.64  
1         NaN         NaN         NaN          NaN       9100.87  
2         NaN         NaN         NaN          NaN       9335.20  
3         NaN         NaN         NaN          NaN       9882.16  
4         NaN         NaN         NaN          NaN      10421.33  
5         NaN         NaN         NaN          NaN        

In [206]:
tata_clean_rows = TATA_clean.dropna(subset=['Sales'])
tata_clean_rows['EBIT_margins'] = tata_clean_rows['EBIT'] / tata_clean_rows['Sales']
tata_clean_rows['CAPEX_ratio'] = abs(tata_clean_rows['CAPEX']) / tata_clean_rows['Sales']
tata_clean_rows['WC_Change'] = tata_clean_rows['Working capital'].diff().fillna(0)
tata_clean_rows['wc_change_ratio'] = tata_clean_rows['WC_Change'] / tata_clean_rows['Sales']
tata_clean_rows['depr_ratio'] = tata_clean_rows['Depreciation'] / tata_clean_rows['Sales']
avg_ebit_margins = tata_clean_rows['EBIT_margins'].mean()
avg_capex_ratio = tata_clean_rows['CAPEX_ratio'].mean()
avg_wc_change_ratio = tata_clean_rows['wc_change_ratio'].mean()
avg_depr_ratio = tata_clean_rows['depr_ratio'].mean() 
tax_rate = 0.25
sales_growth = 0.06
projected_metrics = {}
sales_2025 = tata_clean_rows.loc[tata_clean_rows['years'] == 2025.0, 'Sales'].values[0]
for i in range(2026, 2031):
    sales_2025 *= (1 + sales_growth)
    projected_EBIT = sales_2025 * avg_ebit_margins
    projected_depre = sales_2025 * avg_depr_ratio
    projected_CAPEX = sales_2025 * avg_capex_ratio
    projected_wc_change = sales_2025 * avg_wc_change_ratio
    projected_EBIAT = projected_EBIT * (1 - tax_rate)
    free_cash_flow = projected_EBIAT + projected_depre - projected_CAPEX - projected_wc_change
    projected_metrics[f"Year {i}"] = {
        "Projected Revenue": round(sales_2025, 2),
        "Operating EBIT": round(projected_EBIT, 2),
        "EBIAT": round(projected_EBIAT, 2),
        "Depreciation Addback": round(projected_depre, 2),
        "CapEx Outflow": round(projected_CAPEX, 2),
        "WC Change Impact": round(projected_wc_change, 2),
        "Free Cash Flow (FCF)": round(free_cash_flow, 2)
    }

projected_FCF = pd.DataFrame(projected_metrics)
print(projected_FCF)
    

    


                      Year 2026  Year 2027  Year 2028  Year 2029  Year 2030
Projected Revenue     231655.58  245554.91  260288.21  275905.50  292459.83
Operating EBIT         26224.28   27797.74   29465.60   31233.54   33107.55
EBIAT (Net Profit)     19668.21   20848.30   22099.20   23425.15   24830.66
Depreciation Addback   10446.83   11073.64   11738.06   12442.35   13188.89
CapEx Outflow          14093.27   14938.87   15835.20   16785.31   17792.43
WC Change Impact       -3989.09   -4228.43   -4482.14   -4751.07   -5036.13
Free Cash Flow (FCF)   20010.86   21211.52   22484.21   23833.26   25263.25


In [207]:
df_tata_raw = yf.download("TATASTEEL.NS", start='2021-01-01', end='2025-12-30', multi_level_index=False)
df_nifty_raw = yf.download("^NSEI", start='2021-01-01', end='2025-12-01', multi_level_index=False)
tata_returns = df_tata_raw['Close'].pct_change()
nifty_returns = df_nifty_raw['Close'].pct_change()
aligned_returns = pd.DataFrame({
    'Tata': tata_returns,
    'Market': nifty_returns
}).dropna() 
clean_tata = aligned_returns['Tata'].values
clean_market = aligned_returns['Market'].values
cov_matrix = np.cov(clean_tata, clean_market)
covariance_stock_market = cov_matrix[0, 1]
variance_market = cov_matrix[1, 1]
beta = covariance_stock_market / variance_market

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [208]:

print(beta)

1.3455262309350657


In [209]:
interest_expense=TATA_clean['Interest'].dropna()
print(interest_expense)

0    7607.0
1    5462.0
2    6299.0
3    7508.0
4    7341.0
Name: Interest, dtype: float64


In [264]:
market_cap = 248422.0     
total_debt = 92382.0      
total_capital = market_cap + total_debt
weight_equity = market_cap / total_capital
weight_debt = total_debt / total_capital
rf_rate = 0.07            
beta = 1.35         

equity_risk_premium = 0.06 
cost_of_equity = rf_rate + (beta * equity_risk_premium)
latest_interest_expense = interest_expense.iloc[-1] 
pre_tax_cost_of_debt =float( latest_interest_expense) /float( total_debt)
after_tax_cost_of_debt = pre_tax_cost_of_debt * (1 - tax_rate)
WACC = (weight_equity * cost_of_equity) + (weight_debt * after_tax_cost_of_debt)

print(f"Capital Weights  Equity: {weight_equity:.2%}, Debt: {weight_debt:.2%}")
print(f"Cost of Equity (Ke)  {cost_of_equity:.3%}")
print(f"Pre-Tax Debt Cost    {pre_tax_cost_of_debt:.3%}")
print(f"After-Tax Debt  {after_tax_cost_of_debt:.2%}")
print(f"WACC    {WACC:.2%}")

Capital Weights  Equity: 72.89%, Debt: 27.11%
Cost of Equity (Ke)  15.100%
Pre-Tax Debt Cost    7.946%
After-Tax Debt  5.96%
WACC    12.62%


In [212]:
print(latest_interest_expense)

7341.0


In [216]:
projected_FCF.head(7)

,Year 2026,Year 2027,Year 2028,Year 2029,Year 2030
Projected Revenue,231655.58,245554.91,260288.21,275905.50,292459.83
Operating EBIT,26224.28,27797.74,29465.60,31233.54,33107.55
EBIAT (Net Profit),19668.21,20848.30,22099.20,23425.15,24830.66
Depreciation Addback,10446.83,11073.64,11738.06,12442.35,13188.89
CapEx Outflow,14093.27,14938.87,15835.20,16785.31,17792.43
WC Change Impact,-3989.09,-4228.43,-4482.14,-4751.07,-5036.13
Free Cash Flow (FCF),20010.86,21211.52,22484.21,23833.26,25263.25


In [219]:
terminal_growth_rate=0.045
discounted_fcf=[]
fcf_values=projected_FCF.loc["Free Cash Flow (FCF)"].values
for i,fcf in enumerate(fcf_values):
    year=i+1
    present_value_fcf = fcf / ((1+WACC)**year)
    discounted_fcf.append(present_value_fcf)
    print(f"year {year} present value: {present_value_fcf:.2f}")


year 1 present value: 17768.11
year 2 present value: 16723.32
year 3 present value: 15739.97
year 4 present value: 14814.44
year 5 present value: 13943.32


In [224]:
fcf_year_5 = fcf_values[-1]
terminal_value = (fcf_year_5 * (1 + terminal_growth_rate)) / (wacc - terminal_growth_rate)
pv_terminal_value = terminal_value / ((1 + wacc) ** 5)
print(f"Raw Terminal Value (At Year 5)    : {terminal_value:,.2f}")
print(f"Present Value of Terminal Value   : {pv_terminal_value:,.2f} ")

Raw Terminal Value (At Year 5)    : 325,030.29
Present Value of Terminal Value   : 179,391.11 


In [231]:
sum_pv_explicit = sum(discounted_fcf)
enterprise_value=sum_pv_explicit+pv_terminal_value
current_market_price = 198.96       
total_debt = 92354.0               
cash_and_equivalents = 12210.0     
shares_outstanding_crores = 1248.35
current_market_price = 198.96
equity_value = enterprise_value - total_debt + cash_and_equivalents
intrinsic_value_per_share = equity_value / shares_outstanding_crores
valuation_gap = ((intrinsic_value_per_share - current_market_price) / current_market_price) * 100

if intrinsic_value_per_share > current_market_price:
    valuation_verdict = "UNDERVALUED (Potential Buy Signal)"
else:
    valuation_verdict = "OVERVALUED (Premium Market Pricing)"
    
print(f"Total Enterprise Value (EV)     : {enterprise_value:,.2f} Cr.")
print(f"(-) Total Debt Balance          :  {total_debt:,.2f} Cr.")
print(f"(+) Cash & Equivalents Balance   : {cash_and_equivalents:,.2f} Cr.")
print(f"TOTAL EQUITY VALUE             :  {equity_value:,.2f} Cr.")
print(f"Divided by Shares Outstanding  : {shares_outstanding_crores} Cr.")
print(f"CALCULATED INTRINSIC VALUE     : {intrinsic_value_per_share:.2f} per Share")
print(f"CURRENT LIVE MARKET PRICE      : {current_market_price:.2f} per Share")
print(f"VALUATION GAP SYSTEM ANALYSIS   : {valuation_gap:+.2f}%")
print(f"FINAL MODEL INVESTMENT VERDICT  : {valuation_verdict}")

Total Enterprise Value (EV)     : 258,380.27 Cr.
(-) Total Debt Balance          :  92,354.00 Cr.
(+) Cash & Equivalents Balance   : 12,210.00 Cr.
TOTAL EQUITY VALUE             :  178,236.27 Cr.
Divided by Shares Outstanding  : 1248.35 Cr.
CALCULATED INTRINSIC VALUE     : 142.78 per Share
CURRENT LIVE MARKET PRICE      : 198.96 per Share
VALUATION GAP SYSTEM ANALYSIS   : -28.24%
FINAL MODEL INVESTMENT VERDICT  : OVERVALUED (Premium Market Pricing)


In [244]:
Infosys = pd.read_csv("INFOSYS.csv")
INFOSYS=pd.DataFrame(Infosys)

print(INFOSYS)

    years     Sales  Working capital      PBT  Interest     EBIT   CAPEX  \
0  2021.0  100472.0          23224.0  26628.0     195.0  26823.0 -7373.0   
1  2022.0  121641.0          23885.0  30110.0     200.0  30310.0 -6485.0   
2  2023.0  146767.0          22467.0  33322.0     284.0  33606.0 -1071.0   
3  2024.0  153670.0          25210.0  35988.0     470.0  36458.0 -5093.0   
4  2025.0  162990.0          35694.0  37608.0     416.0  38024.0 -1864.0   
5     NaN       NaN              NaN      NaN       NaN      NaN     NaN   

   Depreciation  
0        3266.0  
1        4182.0  
2        4225.0  
3        4678.0  
4        4812.0  
5           NaN  


In [246]:
INFO_clean_rows = INFOSYS.dropna(subset=['Sales'])
INFO_clean_rows['EBIT_margins'] = INFO_clean_rows['EBIT'] / INFO_clean_rows['Sales']
INFO_clean_rows['CAPEX_ratio'] = abs(INFO_clean_rows['CAPEX']) / INFO_clean_rows['Sales']
INFO_clean_rows['WC_Change'] = INFO_clean_rows['Working capital'].diff().fillna(0)
INFO_clean_rows['wc_change_ratio'] = INFO_clean_rows['WC_Change'] / INFO_clean_rows['Sales']
INFO_clean_rows['depr_ratio'] = INFO_clean_rows['Depreciation'] / INFO_clean_rows['Sales']
avg_ebit_margins_2 = INFO_clean_rows['EBIT_margins'].mean()
avg_capex_ratio_2 = INFO_clean_rows['CAPEX_ratio'].mean()
avg_wc_change_ratio_2 = INFO_clean_rows['wc_change_ratio'].mean()
avg_depr_ratio_2= INFO_clean_rows['depr_ratio'].mean() 
tax_rate = 0.25
sales_growth_2 = 0.06
projected_metrics_2 = {}
sales_2025_2 = INFO_clean_rows.loc[tata_clean_rows['years'] == 2025.0, 'Sales'].values[0]
for i in range(2026, 2031):
    sales_2025_2 *= (1 + sales_growth_2)
    projected_EBIT_2 = sales_2025_2 * avg_ebit_margins_2
    projected_depre_2 = sales_2025_2 * avg_depr_ratio_2
    projected_CAPEX_2 = sales_2025_2 * avg_capex_ratio_2
    projected_wc_change_2 = sales_2025_2 * avg_wc_change_ratio_2
    projected_EBIAT_2= projected_EBIT_2 * (1 - tax_rate)
    free_cash_flow_2 = projected_EBIAT_2 + projected_depre_2 - projected_CAPEX_2 - projected_wc_change_2
    projected_metrics_2[f"Year {i}"] = {
        "Projected Revenue": round(sales_2025_2, 2),
        "Operating EBIT": round(projected_EBIT_2, 2),
        "EBIAT": round(projected_EBIAT_2, 2),
        "Depreciation Addback": round(projected_depre_2, 2),
        "CapEx Outflow": round(projected_CAPEX_2, 2),
        "WC Change Impact": round(projected_wc_change_2, 2),
        "Free Cash Flow (FCF)": round(free_cash_flow_2, 2)
    }

projected_FCF_2 = pd.DataFrame(projected_metrics_2)
print(projected_FCF_2)
    

    


                      Year 2026  Year 2027  Year 2028  Year 2029  Year 2030
Projected Revenue     172769.40  183135.56  194123.70  205771.12  218117.39
Operating EBIT         42005.77   44526.11   47197.68   50029.54   53031.32
EBIAT                  31504.33   33394.59   35398.26   37522.16   39773.49
Depreciation Addback    5377.92    5700.60    6042.63    6405.19    6789.50
CapEx Outflow           6170.36    6540.59    6933.02    7349.00    7789.94
WC Change Impact        2693.31    2854.91    3026.21    3207.78    3400.25
Free Cash Flow (FCF)   28018.57   29699.68   31481.66   33370.56   35372.80


In [255]:
df_INFO_raw = yf.download("INFY.NS", start='2021-01-01', end='2025-12-30', multi_level_index=False)
df_nifty_raw = yf.download("^NSEI", start='2021-01-01', end='2025-12-01', multi_level_index=False)
info_returns = df_INFO_raw['Close'].pct_change()
nifty_returns = df_nifty_raw['Close'].pct_change()
aligned_returns_2 = pd.DataFrame({
    'info': info_returns,
    'Market': nifty_returns
}).dropna() 
clean_info = aligned_returns_2['info'].values
clean_market = aligned_returns_2['Market'].values
cov_matrix_2= np.cov(clean_info, clean_market)
covariance_stock_market_2 = cov_matrix_2[0, 1]
variance_market_2 = cov_matrix_2[1, 1]
beta_2 = covariance_stock_market_2 / variance_market_2

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [259]:
print(clean_info)

[ 0.02205566  0.00430822 -0.00904309 ...  0.01783617  0.00545605
 -0.004022  ]


In [256]:
print(beta_2)

0.9586395887070658


In [262]:
interest_expense_2=INFO_clean_rows['Interest'].dropna()
print(interest_expense_2)

0    195.0
1    200.0
2    284.0
3    470.0
4    416.0
Name: Interest, dtype: float64


In [263]:
market_cap_2 = 507192.0       
total_debt_2 = 8222.0         
tax_rate = 0.25     
total_capital_2 = market_cap + total_debt
weight_equity_2= market_cap / total_capital
weight_debt_2= total_debt / total_capital
rf_rate = 0.07            
beta_2 = 0.959         

equity_risk_premium = 0.06 
cost_of_equity_2 = rf_rate + (beta_2 * equity_risk_premium)
latest_interest_expense_2 = interest_expense_2.iloc[-1] 
pre_tax_cost_of_debt_2 =float( latest_interest_expense_2) /float( total_debt_2)
after_tax_cost_of_debt_2 = pre_tax_cost_of_debt_2 * (1 - tax_rate)
WACC_2 = (weight_equity_2 * cost_of_equity_2) + (weight_debt_2 * after_tax_cost_of_debt_2)

print(f"Capital Weights  Equity: {weight_equity_2:.2%}, Debt: {weight_debt_2:.2%}")
print(f"Cost of Equity (Ke)  {cost_of_equity_2:.3%}")
print(f"Pre-Tax Debt Cost    {pre_tax_cost_of_debt_2:.3%}")
print(f"After-Tax Debt  {after_tax_cost_of_debt_2:.2%}")
print(f"WACC    {WACC_2:.2%}")

Capital Weights  Equity: 72.89%, Debt: 27.10%
Cost of Equity (Ke)  12.754%
Pre-Tax Debt Cost    5.060%
After-Tax Debt  3.79%
WACC    10.33%


In [265]:
print(latest_interest_expense_2)

416.0


In [266]:
projected_FCF_2.head(7)

,Year 2026,Year 2027,Year 2028,Year 2029,Year 2030
Projected Revenue,172769.40,183135.56,194123.70,205771.12,218117.39
Operating EBIT,42005.77,44526.11,47197.68,50029.54,53031.32
EBIAT,31504.33,33394.59,35398.26,37522.16,39773.49
Depreciation Addback,5377.92,5700.60,6042.63,6405.19,6789.50
CapEx Outflow,6170.36,6540.59,6933.02,7349.00,7789.94
WC Change Impact,2693.31,2854.91,3026.21,3207.78,3400.25
Free Cash Flow (FCF),28018.57,29699.68,31481.66,33370.56,35372.80


In [268]:
terminal_growth_rate=0.045
discounted_fcf_2=[]
fcf_values_2=projected_FCF_2.loc["Free Cash Flow (FCF)"].values
for i,fcf in enumerate(fcf_values_2):
    year=i+1
    present_value_fcf_2 = fcf / ((1+WACC_2)**year)
    discounted_fcf_2.append(present_value_fcf)
    print(f"year {year} present value: {present_value_fcf_2:.2f}")

year 1 present value: 25396.37
year 2 present value: 24400.75
year 3 present value: 23444.17
year 4 present value: 22525.09
year 5 present value: 21642.04


In [270]:
Fcf_year_5 = fcf_values_2[-1]
terminal_value_2 = (Fcf_year_5 * (1 + terminal_growth_rate)) / (WACC_2 - terminal_growth_rate)
pv_terminal_value_2 = terminal_value_2 / ((1 + WACC_2) ** 5)
print(f"Raw Terminal Value (At Year 5)    : {terminal_value_2:,.2f}")
print(f"Present Value of Terminal Value   : {pv_terminal_value_2:,.2f} ")

Raw Terminal Value (At Year 5)    : 634,575.93
Present Value of Terminal Value   : 388,250.73 


In [272]:
sum_pv_explicit_2 = sum(discounted_fcf_2)
enterprise_value_2=sum_pv_explicit_2+pv_terminal_value_2
current_market_price_2 = 1062.10      
total_debt_2 = 9119.00             
cash_and_equivalents_2 = 43075.00    
shares_outstanding_crores_2 = 1248.35
current_market_price_2 = 404.69
equity_value_2 = enterprise_value_2 - total_debt_2 + cash_and_equivalents_2
intrinsic_value_per_share_2 = equity_value_2 / shares_outstanding_crores_2
valuation_gap_2 = ((intrinsic_value_per_share_2 - current_market_price_2) / current_market_price_2) * 100

if intrinsic_value_per_share_2 > current_market_price_2:
    valuation_verdict_2 = "UNDERVALUED (Potential Buy Signal)"
else:
    valuation_verdict_2= "OVERVALUED (Premium Market Pricing)"
    
print(f"Total Enterprise Value (EV)     : {enterprise_value_2:,.2f} Cr.")
print(f"(-) Total Debt Balance          :  {total_debt_2:,.2f} Cr.")
print(f"(+) Cash & Equivalents Balance   : {cash_and_equivalents_2:,.2f} Cr.")
print(f"TOTAL EQUITY VALUE             :  {equity_value_2:,.2f} Cr.")
print(f"Divided by Shares Outstanding  : {shares_outstanding_crores_2} Cr.")
print(f"CALCULATED INTRINSIC VALUE     : {intrinsic_value_per_share_2:.2f} per Share")
print(f"CURRENT LIVE MARKET PRICE      : {current_market_price_2:.2f} per Share")
print(f"VALUATION GAP SYSTEM ANALYSIS   : {valuation_gap_2:+.2f}%")
print(f"FINAL MODEL INVESTMENT VERDICT  : {valuation_verdict_2}")

Total Enterprise Value (EV)     : 457,967.35 Cr.
(-) Total Debt Balance          :  9,119.00 Cr.
(+) Cash & Equivalents Balance   : 43,075.00 Cr.
TOTAL EQUITY VALUE             :  491,923.35 Cr.
Divided by Shares Outstanding  : 1248.35 Cr.
CALCULATED INTRINSIC VALUE     : 394.06 per Share
CURRENT LIVE MARKET PRICE      : 404.69 per Share
VALUATION GAP SYSTEM ANALYSIS   : -2.63%
FINAL MODEL INVESTMENT VERDICT  : OVERVALUED (Premium Market Pricing)


In [273]:
print(TATA_clean)

    years     Sales    CAPEX  Working capital      PBT  Interest     EBIT  \
0  2021.0  156477.0  -9437.0          44327.0  13844.0    7607.0  21451.0   
1  2022.0  243959.0 -10905.0          44381.0  50227.0    5462.0  55689.0   
2  2023.0  243353.0 -18179.0          21683.0  18235.0    6299.0  24534.0   
3  2024.0  229171.0 -14253.0          20301.0  -1147.0    7508.0   6361.0   
4  2025.0  218543.0 -13611.0          23138.0   8413.0    7341.0  15754.0   
5     NaN       NaN      NaN              NaN      NaN       NaN      NaN   

   Unnamed: 7  Unnamed: 8  Unnamed: 9  Unnamed: 10  Depreciation  
0         NaN         NaN         NaN          NaN       9233.64  
1         NaN         NaN         NaN          NaN       9100.87  
2         NaN         NaN         NaN          NaN       9335.20  
3         NaN         NaN         NaN          NaN       9882.16  
4         NaN         NaN         NaN          NaN      10421.33  
5         NaN         NaN         NaN          NaN        

Intrpretation:
Positive Growth Cash Generation:Free Cash Flows (FCF) are projected to grow systematically from 20,010.86 Cr (2026) to 25,263.25 Cr. (2030), driven by strong operational EBIAT and steady depreciation addbacks.
Risk-Adjusted Hurdle Rate: Applying an equity weight of 72.89% K_e = 15.10% and an after-tax debt weight of 27.11% K_d = 5.96% yields a blended WACC of 12.62% to discount future operational cash flows.
Enterprise-to-Equity Bridge:** Discounting the firm's cash flows establishes a Total Enterprise Value of 258,380.27 Cr., which adjusts to a Total Equity Value of 178,236.27 Cr. after accounting for net debt 92,354 Cr. debt less 12,210 Cr. cash).
Significant Overvaluation Identified:Dividing equity value by 1,248.35 Cr. outstanding shares computes an intrinsic baseline value of 142.78 per share, contrasting heavily with the current live market price of 198.96.
Negative Valuation Discrepancy: The asset is currently trading at a sharp -28.24% valuation gap, indicating that the public market is pricing in a massive premium above fundamentals.
Definitive Investment Verdict: The valuation model issues an unambiguous OVERVALUED stance, warning against long positions at current price levels due to compressed safety margins.
